## __Generación customer journey data__ 

In [7]:
import pandas as pd
import numpy as np
import uuid
import csv
from datetime import datetime, timedelta

# ==========================================
# 1. CONFIGURACIÓN Y MAESTROS
# ==========================================
num_rows_target = 2100000
output_file = 'data_clase_sql_final_mejorado.csv'
np.random.seed(42)

paises_config = {
    'México': {'share': 0.40, 'mult': 0.90},
    'España': {'share': 0.25, 'mult': 1.25},
    'Colombia': {'share': 0.15, 'mult': 0.85},
    'Argentina': {'share': 0.12, 'mult': 1.00},
    'Chile': {'share': 0.08, 'mult': 1.10}
}

usuarios_pool = np.arange(10000, 99999)
user_geo = {u: np.random.choice(list(paises_config.keys()), p=[c['share'] for c in paises_config.values()]) for u in usuarios_pool}
skus_pool = [f"SKU-{i}" for i in range(100, 1000)]

print(f"Generando {num_rows_target:,} registros en formato YYYY-MM-DD HH:MM:SS...")

# ==========================================
# 2. GENERACIÓN CON FORMATO ISO 8601
# ==========================================
headers = ['event_timestamp', 'user_id', 'session_id', 'event_name', 'country', 'traffic_source', 'device', 'sku', 'cart_id', 'payment_method', 'units', 'amount']
date_fmt = '%Y-%m-%d %H:%M:%S' # <--- Formato Estándar Definido

with open(output_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    
    current_rows = 0
    while current_rows < num_rows_target:
        uid = int(np.random.choice(usuarios_pool))
        pais = user_geo[uid]
        mult = paises_config[pais]['mult']
        sid = str(uuid.uuid4())[:18].upper()
        ts = datetime(2026, 1, 1) + timedelta(seconds=np.random.randint(0, 2678400))
        
        src = np.random.choice(['Google Search', 'Direct', 'Social Media', 'Email Campaign'], p=[0.25, 0.20, 0.50, 0.05])
        dev = np.random.choice(['Mobile', 'Desktop', 'Tablet'], p=[0.6, 0.3, 0.1])
        
        session_rows = []
        # PASO 1: Home
        session_rows.append([ts.strftime(date_fmt), uid, sid, 'home_page', pais, src, dev, None, None, None, 0, 0.0])
        
        # PASO 2: Login
        if np.random.random() < 0.75:
            ts += timedelta(seconds=20)
            session_rows.append([ts.strftime(date_fmt), uid, sid, 'login', pais, src, dev, None, None, None, 0, 0.0])
            
            # PASO 3: Product View
            if np.random.random() < 0.80:
                for _ in range(np.random.randint(1, 4)):
                    ts += timedelta(minutes=2)
                    session_rows.append([ts.strftime(date_fmt), uid, sid, 'product_view', pais, src, dev, np.random.choice(skus_pool), None, None, 0, 0.0])
                
                # PASO 4: Add to Cart
                conv_base = 0.60 if src == 'Email Campaign' else 0.35
                if np.random.random() < conv_base:
                    cart_id_fijo = f"CART-{np.random.randint(10000, 99999)}"
                    pago_fijo = np.random.choice(['Credit Card', 'PayPal', 'Debit Card', 'Crypto'])
                    skus_compra = np.random.choice(skus_pool, size=np.random.randint(1, 3), replace=False)
                    
                    for sku in skus_compra:
                        ts_step = ts + timedelta(seconds=30)
                        session_rows.append([ts_step.strftime(date_fmt), uid, sid, 'add_to_cart', pais, src, dev, sku, cart_id_fijo, None, 0, 0.0])
                        
                        # PASO 5: Begin Checkout
                        if np.random.random() < 0.85:
                            ts_step += timedelta(seconds=10)
                            session_rows.append([ts_step.strftime(date_fmt), uid, sid, 'begin_checkout', pais, src, dev, sku, cart_id_fijo, None, 0, 0.0])
                            
                            # PASO 6: End Checkout
                            if np.random.random() < 0.90:
                                ts_step += timedelta(seconds=10)
                                session_rows.append([ts_step.strftime(date_fmt), uid, sid, 'end_checkout', pais, src, dev, sku, cart_id_fijo, None, 0, 0.0])
                                
                                # PASO 7: Purchase
                                if np.random.random() < 0.95:
                                    ts_step += timedelta(seconds=10)
                                    u, a = np.random.randint(1, 3), round(np.random.uniform(40, 150) * mult, 2)
                                    session_rows.append([ts_step.strftime(date_fmt), uid, sid, 'purchase', pais, src, dev, sku, cart_id_fijo, pago_fijo, u, a])

        for r in session_rows:
            writer.writerow(r)
        current_rows += len(session_rows)

# ==========================================
# 3. SANEAMIENTO Y EXPORTACIÓN FINAL
# ==========================================
print("Saneando integridad de pagos y forzando tipos...")
# Leemos con parse_dates para evitar el ValueError en este paso
df_final = pd.read_csv(output_file, parse_dates=['event_timestamp'])

# Asegurar un solo método de pago por carrito
fix_pago = df_final[df_final['event_name'] == 'purchase'].groupby('cart_id')['payment_method'].first()
df_final.loc[df_final['event_name'] == 'purchase', 'payment_method'] = df_final.loc[df_final['event_name'] == 'purchase', 'cart_id'].map(fix_pago)

# Guardar con el formato final deseado
df_final.to_csv(output_file, index=False, date_format='%Y-%m-%d %H:%M:%S')

# ========================================== 
# 4. REPORTE DE CALIDAD
# ==========================================  
print("\n" + "="*40)
print(f"Dataset generado: {output_file}")
print(f"Muestra de fecha: {df_final['event_timestamp'].iloc[0]}")
print(f"Errores de pago: {df_final[df_final['event_name'] == 'purchase'].groupby('cart_id')['payment_method'].nunique().gt(1).sum()}")
print("="*40)

Generando 2,100,000 registros en formato YYYY-MM-DD HH:MM:SS...
Saneando integridad de pagos y forzando tipos...

Dataset generado: data_clase_sql_final_mejorado.csv
Muestra de fecha: 2026-01-11 05:48:18
Errores de pago: 0


In [8]:
df_final.head()

,event_timestamp,user_id,session_id,event_name,country,traffic_source,device,sku,cart_id,payment_method,units,amount
0,2026-01-11 05:48:18,17176,6567177D-1AB5-41C0,home_page,Chile,Google Search,Desktop,NaN,NaN,NaN,0,0.0
1,2026-01-11 05:48:38,17176,6567177D-1AB5-41C0,login,Chile,Google Search,Desktop,NaN,NaN,NaN,0,0.0
2,2026-01-11 05:50:38,17176,6567177D-1AB5-41C0,product_view,Chile,Google Search,Desktop,SKU-758,NaN,NaN,0,0.0
3,2026-01-11 05:52:38,17176,6567177D-1AB5-41C0,product_view,Chile,Google Search,Desktop,SKU-656,NaN,NaN,0,0.0
4,2026-01-11 05:54:38,17176,6567177D-1AB5-41C0,product_view,Chile,Google Search,Desktop,SKU-474,NaN,NaN,0,0.0


#### __Validación data funnel__

In [ ]:
usuarios_unicos_por_evento = df_final.groupby('event_name')['user_id'].nunique().sort_values(ascending=False)
print(usuarios_unicos_por_evento)  

event_name
home_page         89735
login             88848
product_view      87184
add_to_cart       64474
begin_checkout    61550
end_checkout      59382
purchase          58163
Name: user_id, dtype: int64


In [11]:
df_final.shape

(2100000, 12)

In [12]:
def validate_ecommerce_data(file_path):
    print(f"--- Iniciando Auditoría de Coherencia para: {file_path} ---\n")
    df = pd.read_csv(file_path)
    
    # 1. Validación de Montos (Solo en compras)
    check_amount = df[(df['event_name'] != 'purchase') & (df['amount'] > 0)]
    val1 = "PASSED" if len(check_amount) == 0 else f"FAILED ({len(check_amount)} errores)"
    print(f"1. Montos fuera de evento 'purchase': {val1}")

    # 2. Validación de Unicidad de Session ID
    # Verificamos que un session_id pertenezca a un solo user_id
    session_user_count = df.groupby('session_id')['user_id'].nunique()
    val2 = "PASSED" if (session_user_count == 1).all() else "FAILED (Sesiones compartidas entre usuarios)"
    print(f"2. Integridad Session-User (1:1): {val2}")

    # 3. Validación de Proporción Poblacional
    # México debe ser > España > Chile
    country_counts = df['country'].value_counts()
    is_proportional = (country_counts['México'] > country_counts['España']) and \
                      (country_counts['España'] > country_counts['Chile'])
    val3 = "PASSED" if is_proportional else "FAILED (Distribución plana detectada)"
    print(f"3. Jerarquía Poblacional Realista: {val3}")

    # 4. Validación de Unidades
    check_units = df[(df['event_name'] == 'purchase') & (df['units'] <= 0)]
    val4 = "PASSED" if len(check_units) == 0 else f"FAILED ({len(check_units)} compras sin unidades)"
    print(f"4. Unidades mínimas en compras (>0): {val4}")

    

    # 6. Validación de Sesión vs Login
    # Toda compra debería haber tenido un evento de 'login' previo en la misma sesión
    purchases_sessions = df[df['event_name'] == 'purchase']['session_id'].unique()
    sessions_with_login = df[df['event_name'] == 'login']['session_id'].unique()
    missing_login = set(purchases_sessions) - set(sessions_with_login)
    val6 = "PASSED" if len(missing_login) == 0 else f"FAILED ({len(missing_login)} sesiones compraron sin login)"
    print(f"6. Flujo de Login antes de compra: {val6}")

    # 7. Validación de Product Views por Sesión (Campana de Gauss)
    avg_views = df[df['event_name'] == 'product_view'].groupby('session_id').size().mean()
    val7 = "PASSED" if 3.5 <= avg_views <= 4.5 else f"FAILED (Media: {avg_views:.2f})"
    print(f"7. Distribución Normal de Views (~4): {val7}")

    # 8. Validación de Valores Nulos (Missing Values)
    critical_cols = ['event_timestamp', 'user_id', 'session_id', 'event_name', 'country']
    nulls = df[critical_cols].isnull().sum().sum()
    val8 = "PASSED" if nulls == 0 else f"FAILED ({nulls} nulos detectados)"
    print(f"8. Integridad de Columnas Críticas (Not Null): {val8}")

    # 9. Validación de Cart ID
    # Verificamos que el cart_id esté presente en eventos de checkout y purchase
    cart_check = df[df['event_name'].isin(['add_to_cart', 'purchase'])]['cart_id'].isnull().sum()
    val9 = "PASSED" if cart_check == 0 else "FAILED (Eventos de carrito sin Cart ID)"
    print(f"9. Consistencia de Cart ID en el funnel: {val9}")

    # 10. Validación de Formato de Tiempo
    try:
        pd.to_datetime(df['event_timestamp'])
        val10 = "PASSED"
    except:
        val10 = "FAILED (Formato de fecha inválido)"
    print(f"10. Integridad de Event Timestamp: {val10}")

    print("\n--- Auditoría Finalizada ---")

# Ejecución
validate_ecommerce_data('data_clase_sql_final_mejorado.csv') 

--- Iniciando Auditoría de Coherencia para: data_clase_sql_final_mejorado.csv ---

1. Montos fuera de evento 'purchase': PASSED
2. Integridad Session-User (1:1): PASSED
3. Jerarquía Poblacional Realista: PASSED
4. Unidades mínimas en compras (>0): PASSED
6. Flujo de Login antes de compra: PASSED
7. Distribución Normal de Views (~4): FAILED (Media: 2.00)
8. Integridad de Columnas Críticas (Not Null): PASSED
9. Consistencia de Cart ID en el funnel: PASSED
10. Integridad de Event Timestamp: PASSED

--- Auditoría Finalizada ---


In [13]:
df_final.dtypes

event_timestamp    datetime64[ns]
user_id                     int64
session_id                 object
event_name                 object
country                    object
traffic_source             object
device                     object
sku                        object
cart_id                    object
payment_method             object
units                       int64
amount                    float64
dtype: object

In [14]:
def generar_tablas_analisis(file_path):
    df = pd.read_csv(file_path)
    
    # Pre-cálculo de órdenes (solo eventos de compra)
    df_compras = df[df['event_name'] == 'purchase']

    # --- TABLA 1: MÉTRICAS POR PAÍS (Validación de Población y Valor) ---
    print("\n1. TABLA POR PAÍS (Volumen y Negocio)")
    tabla_pais = df.groupby('country').agg(
        total_eventos=('event_name', 'count'),
        usuarios_unicos=('user_id', 'nunique'),
        sesiones_unicos=('session_id', 'nunique'),
        total_ordenes=('amount', lambda x: (x > 0).sum()),
        ingresos_totales=('amount', 'sum')
    ).sort_values('total_eventos', ascending=False)
    print(tabla_pais)

    # --- TABLA 2: MÉTRICAS POR DISPOSITIVO (Validación de Tecnología) ---
    print("\n2. TABLA POR DISPOSITIVO")
    tabla_device = df.groupby('device').agg(
        total_eventos=('event_name', 'count'),
        sesiones=('session_id', 'nunique'),
        compras=('event_name', lambda x: (x == 'purchase').sum())
    )
    print(tabla_device)

    # --- TABLA 3: SESGO DE TRÁFICO (Validación de Canales) ---
    print("\n3. TABLA POR FUENTE DE TRÁFICO (Efectividad)")
    tabla_source = df.groupby('traffic_source').agg(
        total_eventos=('event_name', 'count'),
        compras=('event_name', lambda x: (x == 'purchase').sum()),
        ticket_promedio=('amount', lambda x: x[x > 0].mean())
    )
    # Calculamos Tasa de Conversión (Compras / Sesiones Únicas de esa fuente)
    tabla_source['conversion_rate'] = (tabla_source['compras'] / df.groupby('traffic_source')['session_id'].nunique() * 100).round(2)
    print(tabla_source.sort_values('conversion_rate', ascending=False))

    # --- TABLA 4: DISTRIBUCIÓN DE EVENTOS (Validación del Funnel) ---
    print("\n4. CONTEO GLOBAL DE EVENTOS (Embudo)")
    tabla_eventos = df['event_name'].value_counts().to_frame()
    print(tabla_eventos)

# Ejecución
generar_tablas_analisis('data_clase_sql_final_mejorado.csv') 


1. TABLA POR PAÍS (Volumen y Negocio)
           total_eventos  usuarios_unicos  sesiones_unicos  total_ordenes  \
country                                                                     
México            840195            35828           207995          49269   
España            525429            22475           130001          30945   
Colombia          315815            13500            78143          18547   
Argentina         253746            10845            62786          14810   
Chile             164815             7087            40862           9603   

           ingresos_totales  
country                      
México           4212361.55  
España           3675587.45  
Colombia         1493575.39  
Argentina        1406451.23  
Chile            1004787.71  

2. TABLA POR DISPOSITIVO
         total_eventos  sesiones  compras
device                                   
Desktop         629141    155674    37005
Mobile         1260453    312012    73845
Tablet          2

In [15]:
def analizar_compras_multiples(file_path):
    print(f"--- Análisis de Órdenes con Múltiples Productos ---")
    df = pd.read_csv(file_path)
    
    # 1. Filtramos solo los eventos de compra
    compras = df[df['event_name'] == 'purchase'].copy()
    
    # 2. Agrupamos por Sesión y Usuario para consolidar la orden
    ordenes = compras.groupby(['session_id', 'user_id']).agg(
        cantidad_skus_distintos=('sku', 'nunique'),
        lista_productos=('sku', lambda x: list(x)),
        total_invertido=('amount', 'sum'),
        total_unidades=('units', 'sum')
    ).reset_index()
    
    # 3. Filtramos los casos donde compraron más de un SKU distinto
    compras_multiples = ordenes[ordenes['cantidad_skus_distintos'] > 1].sort_values(by='cantidad_skus_distintos', ascending=False)
    
    if not compras_multiples.empty:
        print(f"ÉXITO: Se encontraron {len(compras_multiples):,} sesiones con compra de múltiples productos.")
        print("\nTop 10 Casos Detectados (Muestra):")
        # Ajustamos el formato de visualización para ver bien las listas
        pd.set_option('display.max_colwidth', None)
        print(compras_multiples[['user_id', 'session_id', 'cantidad_skus_distintos', 'lista_productos', 'total_invertido']].head(10))
        
        # 4. Estadísticas rápidas para validación de sesgos
        avg_items = ordenes['cantidad_skus_distintos'].mean()
        print(f"\nEstadística Global:")
        print(f"- Promedio de SKUs distintos por orden: {avg_items:.2f}")
    else:
        print("ALERTA: No se detectaron compras de múltiples SKUs. Revisar lógica del generador.")

# Ejecución de la función
analizar_compras_multiples('data_clase_sql_final_mejorado.csv') 

--- Análisis de Órdenes con Múltiples Productos ---
ÉXITO: Se encontraron 29,796 sesiones con compra de múltiples productos.

Top 10 Casos Detectados (Muestra):
       user_id          session_id  cantidad_skus_distintos  \
0        99878  000146A8-2578-41B9                        2   
62497    76495  AB02BE36-40C0-48B5                        2   
62532    41228  AB1A5215-BFB0-4159                        2   
62531    57794  AB1A4F6B-D542-4E99                        2   
62527    21566  AB193244-E35C-4717                        2   
62523    76275  AB155BA3-4F1B-44D1                        2   
62521    45267  AB154C9D-FF0E-4D4F                        2   
62520    62617  AB144726-D746-4F05                        2   
62515    30637  AB104600-D241-4481                        2   
62514    24372  AB0F89A4-FEB2-4F51                        2   

          lista_productos  total_invertido  
0      [SKU-252, SKU-575]           230.06  
62497  [SKU-735, SKU-592]           220.19  
62532  [SK

In [16]:
def validar_compras_multiples(file_path):
    print(f"--- Auditoría de Canastas Multi-Producto ---")
    df = pd.read_csv(file_path)
    
    # 1. Filtramos solo los eventos de compra
    compras = df[df['event_name'] == 'purchase'].copy()
    
    # 2. Agrupamos por usuario y sesión para ver la "Cesta"
    # Concatenamos los SKUs y sumamos montos/unidades
    cestas = compras.groupby(['user_id', 'session_id', 'country']).agg(
        distintos_skus=('sku', 'nunique'),
        lista_skus=('sku', lambda x: list(x)),
        total_unidades=('units', 'sum'),
        total_pagado=('amount', 'sum')
    ).reset_index()
    
    # 3. Filtramos los casos donde compraron 2 o más SKUs diferentes
    multi_sku = cestas[cestas['distintos_skus'] > 1]
    if not multi_sku.empty:
        print(f"ÉXITO: Se encontraron {len(multi_sku):,} sesiones con múltiples productos.")
        print("\nMuestra de los Top 10 casos detectados:")
        # Ajuste para ver la lista completa de SKUs
        pd.set_option('display.max_colwidth', None)
        print(multi_sku[['user_id', 'country', 'distintos_skus', 'lista_skus', 'total_pagado']].head(50))
        
        # Validación de Identidad (Victoria previa)
        print(f"\n--- Verificación de Consistencia Geográfica ---")
        conteo_paises = multi_sku.groupby('user_id')['country'].nunique()
        if conteo_paises.max() == 1:
            print("CONFIRMADO: Cada usuario mantiene un único país en sus compras múltiples.")
    else:
        print("ALERTA: No se detectaron compras múltiples. Revisar lógica del generador.")

# Ejecución
validar_compras_multiples('data_clase_sql_final_mejorado.csv') 

--- Auditoría de Canastas Multi-Producto ---
ÉXITO: Se encontraron 29,796 sesiones con múltiples productos.

Muestra de los Top 10 casos detectados:
     user_id    country  distintos_skus          lista_skus  total_pagado
2      10002   Colombia               2  [SKU-908, SKU-759]        137.34
4      10003     España               2  [SKU-882, SKU-125]        263.51
5      10004     México               2  [SKU-212, SKU-588]        143.85
6      10005     México               2  [SKU-737, SKU-980]        138.83
13     10008     España               2  [SKU-340, SKU-646]        197.19
15     10009   Colombia               2  [SKU-770, SKU-570]        207.06
16     10011      Chile               2  [SKU-264, SKU-518]        193.65
19     10019     México               2  [SKU-414, SKU-651]        193.79
21     10023     México               2  [SKU-367, SKU-862]         99.46
23     10026     México               2  [SKU-643, SKU-515]        176.13
26     10028     España              

In [17]:
df_final[(df_final['user_id'] == 10002)].sort_values(by='event_timestamp')

,event_timestamp,user_id,session_id,event_name,country,traffic_source,device,sku,cart_id,payment_method,units,amount
637500,2026-01-08 12:07:48,10002,2541D81A-067B-47DC,home_page,Colombia,Social Media,Tablet,NaN,NaN,NaN,0,0.00
637501,2026-01-08 12:08:08,10002,2541D81A-067B-47DC,login,Colombia,Social Media,Tablet,NaN,NaN,NaN,0,0.00
637502,2026-01-08 12:10:08,10002,2541D81A-067B-47DC,product_view,Colombia,Social Media,Tablet,SKU-215,NaN,NaN,0,0.00
637503,2026-01-08 12:12:08,10002,2541D81A-067B-47DC,product_view,Colombia,Social Media,Tablet,SKU-728,NaN,NaN,0,0.00
637508,2026-01-08 12:12:38,10002,2541D81A-067B-47DC,add_to_cart,Colombia,Social Media,Tablet,SKU-759,CART-27166,NaN,0,0.00
637504,2026-01-08 12:12:38,10002,2541D81A-067B-47DC,add_to_cart,Colombia,Social Media,Tablet,SKU-908,CART-27166,NaN,0,0.00
637509,2026-01-08 12:12:48,10002,2541D81A-067B-47DC,begin_checkout,Colombia,Social Media,Tablet,SKU-759,CART-27166,NaN,0,0.00
637505,2026-01-08 12:12:48,10002,2541D81A-067B-47DC,begin_checkout,Colombia,Social Media,Tablet,SKU-908,CART-27166,NaN,0,0.00
637506,2026-01-08 12:12:58,10002,2541D81A-067B-47DC,end_checkout,Colombia,Social Media,Tablet,SKU-908,CART-27166,NaN,0,0.00
637510,2026-01-08 12:12:58,10002,2541D81A-067B-47DC,end_checkout,Colombia,Social Media,Tablet,SKU-759,CART-27166,NaN,0,0.00


In [18]:
# 1. Contamos cuántos usuarios únicos llegaron a cada etapa
embudo_real = df_final.groupby('event_name')['user_id'].nunique().sort_values(ascending=False)

# 2. Lo convertimos a DataFrame para que sea más legible
df_embudo = embudo_real.to_frame(name='usuarios_unicos')

# 3. Calculamos el porcentaje de retención respecto al paso anterior
df_embudo['retencion_%'] = (df_embudo['usuarios_unicos'] / df_embudo['usuarios_unicos'].shift(1) * 100).round(2)

print(df_embudo)

                usuarios_unicos  retencion_%
event_name                                  
home_page                 89735          NaN
login                     88848        99.01
product_view              87184        98.13
add_to_cart               64474        73.95
begin_checkout            61550        95.46
end_checkout              59382        96.48
purchase                  58163        97.95


In [19]:
 #1. Cargamos el dataframe que acabas de generar
# Nota: Si ya tienes 'df_final' en memoria, puedes saltar la lectura
if 'df_final' not in locals():
    print("Cargando df_final desde el CSV...")
    df_final = pd.read_csv('data_clase_sql_final_mejorado.csv') 

# 2. Filtramos solo los eventos de compra
df_purchases = df_final[df_final['event_name'] == 'purchase']

# 3. Agrupamos por cart_id y contamos cuántos métodos de pago únicos tiene cada uno 
check_pagos = df_purchases.groupby('cart_id')['payment_method'].nunique().reset_index()

# 4. Identificamos carritos que tengan más de 1 método (el error que teníamos)
carritos_con_error = check_pagos[check_pagos['payment_method'] > 1]

# 5. Output de control
print("-" * 40)
print(f"RESUMEN DE CALIDAD DE PAGOS")
print("-" * 40)
print(f"Total de carritos analizados: {len(check_pagos):,}")
print(f"Carritos con errores de consistencia: {len(carritos_con_error)}")

if len(carritos_con_error) > 0:
    print("\nDetalle de los primeros errores encontrados:")
    print(carritos_con_error.head())
else:
    print("\n✅ VALIDACIÓN EXITOSA: Cada cart_id tiene un único payment_method.")
print("-" * 40)

----------------------------------------
RESUMEN DE CALIDAD DE PAGOS
----------------------------------------
Total de carritos analizados: 58,181
Carritos con errores de consistencia: 0

✅ VALIDACIÓN EXITOSA: Cada cart_id tiene un único payment_method.
----------------------------------------


In [20]:

# 1. Aseguramos que df_final esté cargado con el último CSV generado
print("Cargando df_final para auditoría final...")
df_final = pd.read_csv('data_clase_sql_final_mejorado.csv')

# 2. Validación: Consistencia de Pago por Carrito
# Solo analizamos eventos de 'purchase' donde el cart_id debe tener 1 solo método
audit_pagos = df_final[df_final['event_name'] == 'purchase'].groupby('cart_id')['payment_method'].nunique()
errores_pago = audit_pagos[audit_pagos > 1].count()

# 3. Validación: Integridad de Usuario (1 País por User_ID)
audit_paises = df_final.groupby('user_id')['country'].nunique()
errores_pais = audit_paises[audit_paises > 1].count()

# --- RESULTADOS ---
print("-" * 40)
print(f"REPORTES DE CONSISTENCIA")
print("-" * 40)
print(f"Errores de Pago (Mismo carrito con varios métodos): {errores_pago}")
print(f"Errores de País (Mismo usuario con varios países): {errores_pais}")
print("-" * 40)

if errores_pago == 0 and errores_pais == 0:
    print("✅ CALIDAD CERTIFICADA: El dataset es consistente.")
else:
    print("❌ ATENCIÓN: Se detectaron inconsistencias en la generación.")

Cargando df_final para auditoría final...
----------------------------------------
REPORTES DE CONSISTENCIA
----------------------------------------
Errores de Pago (Mismo carrito con varios métodos): 0
Errores de País (Mismo usuario con varios países): 0
----------------------------------------
✅ CALIDAD CERTIFICADA: El dataset es consistente.


In [21]:
# Cruzamos Fuente de Tráfico vs Dispositivo para ver la distribución
pivot_variabilidad = pd.crosstab(df_final['traffic_source'], df_final['device'], normalize='index') * 100

print("--- Distribución de Dispositivos por Fuente (%) ---")
print(pivot_variabilidad.round(2))

# Verificación de Variabilidad:
# Si todos los números fueran idénticos (ej. todos 60.00), la data sería sospechosa.
# Al haber decimales distintos (ej. 59.82 vs 60.15), confirmamos aleatoriedad orgánica.

--- Distribución de Dispositivos por Fuente (%) ---
device          Desktop  Mobile  Tablet
traffic_source                         
Direct            29.91   59.97   10.12
Email Campaign    29.66   60.19   10.14
Google Search     29.84   60.40    9.76
Social Media      30.08   59.83   10.09


## __Dimensión: PRODUCTS__

In [23]:
# 1. Definición de coherencia (Marca -> Categoría) y (Subcategoría -> Modelos)
import random
category_brands = {
    'Electronics': ['Samsung', 'Apple', 'Sony', 'Xiaomi'],
    'Home & Kitchen': ['Dyson', 'Philips', 'Ninja', 'DeLonghi'],
    'Fashion': ['Nike', 'Adidas', 'Patagonia', 'Columbia'],
    'Beauty': ['L\'Oréal', 'Chanel', 'Estée Lauder']
}

catalog_details = {
    'Electronics': {
        'Smartphones': ['Galaxy S23', 'iPhone 15', 'Pixel 8', 'Redmi Note'],
        'Laptops': ['MacBook Air', 'XPS 13', 'ThinkPad', 'Surface'],
        'Headphones': ['WH-1000XM5', 'AirPods Pro', 'Buds Pro']
    },
    'Home & Kitchen': {
        'Coffee Makers': ['Vertuo', 'K-Elite', 'Magnifica'],
        'Air Fryers': ['Foodi', 'AirFryer XXL', 'Turbo Blaze'],
        'Vacuum Cleaners': ['V15 Detect', 'Roomba j7', 'Stratos']
    },
    'Fashion': {
        'Running Shoes': ['UltraBoost', 'Air Max', 'Clifton', 'Pegasus'],
        'Jackets': ['Nuptse', 'Ascender', 'Down Sweater'],
        'Watches': ['G-Shock', 'Seiko 5', 'Fenix']
    },
    'Beauty': {
        'Skincare': ['Hyaluronic Acid', 'Retinol Cream', 'Vitamin C'],
        'Fragrances': ['Sauvage', 'Chanel No. 5', 'Acqua di Gio']
    }
}

vendors_map = {
    'Electronics': 'TechLogistics Corp',
    'Home & Kitchen': 'Global Home Solutions',
    'Fashion': 'Style & Wear Co.',
    'Beauty': 'Beauty Essentials Ltd'
}

# 2. Construcción de la Dimensión con Coherencia Total
product_rows = []

for i in range(100, 1501):
    sku = f"SKU-{i}"
    # Elegimos categoría y luego marca/subcategoría que PERTENEZCAN a esa categoría
    cat = random.choice(list(catalog_details.keys()))
    brand = random.choice(category_brands[cat])
    sub_cat = random.choice(list(catalog_details[cat].keys()))
    model = random.choice(catalog_details[cat][sub_cat])
    
    # Atributos lógicos
    vendor = vendors_map[cat]
    weight = round(random.uniform(0.1, 5.0) if cat != 'Home & Kitchen' else random.uniform(2.0, 10.0), 2)
    rating = round(random.uniform(3.5, 5.0), 1)

    product_rows.append([
        sku, f"{brand} {model}", cat, sub_cat, brand, vendor, weight, rating
    ])

# 3. Crear DataFrame Final (Sin unit_price ni unit_cost)
df_products = pd.DataFrame(product_rows, columns=[
    'sku', 'product_name', 'category', 'sub_category', 'brand', 'vendor', 'weight_kg', 'product_rating'
])

# Guardar y regenerar df_final (solo para asegurar que lo tienes en memoria)
df_products.to_csv('dim_products.csv', index=False)

print("✅ dim_products.csv corregida:")
print("- Sin columnas de costo/precio.")
print("- Coherencia de marca y categoría total (Ej: Samsung solo en Electronics).")
print(f"Columnas actuales: {list(df_products.columns)}")

✅ dim_products.csv corregida:
- Sin columnas de costo/precio.
- Coherencia de marca y categoría total (Ej: Samsung solo en Electronics).
Columnas actuales: ['sku', 'product_name', 'category', 'sub_category', 'brand', 'vendor', 'weight_kg', 'product_rating']


In [24]:
df_products.head()

,sku,product_name,category,sub_category,brand,vendor,weight_kg,product_rating
0,SKU-100,Adidas Pegasus,Fashion,Running Shoes,Adidas,Style & Wear Co.,3.73,4.4
1,SKU-101,Samsung Galaxy S23,Electronics,Smartphones,Samsung,TechLogistics Corp,4.53,4.2
2,SKU-102,DeLonghi Stratos,Home & Kitchen,Vacuum Cleaners,DeLonghi,Global Home Solutions,5.52,4.5
3,SKU-103,DeLonghi Vertuo,Home & Kitchen,Coffee Makers,DeLonghi,Global Home Solutions,2.94,4.9
4,SKU-104,L'Oréal Acqua di Gio,Beauty,Fragrances,L'Oréal,Beauty Essentials Ltd,0.28,4.6


### __Validación: Dimensión products__

In [26]:
# Unimos los hechos de compra con la dimensión de productos
df_analysis = df_final[df_final['event_name'] == 'purchase'].merge(
    df_products, 
    on='sku', 
    how='inner'
)

print(f"Total de registros de compra con información de producto: {len(df_analysis):,}")

Total de registros de compra con información de producto: 123,174


In [27]:
# Conteo de ventas por país y categoría
coherencia_cat = df_analysis.groupby(['country', 'category']).size().unstack(fill_value=0)   

print("Distribución de Ventas por País y Categoría:")
print(coherencia_cat)

Distribución de Ventas por País y Categoría:
category   Beauty  Electronics  Fashion  Home & Kitchen
country                                                
Argentina    3942         3817     3375            3676
Chile        2496         2486     2291            2330
Colombia     4978         4743     4342            4484
España       8152         8089     7160            7544
México      13117        12708    11527           11917


In [28]:
# Top 5 marcas con más unidades vendidas por país
top_brands = df_analysis.groupby(['country', 'brand'])['units'].sum().reset_index()
top_brands = top_brands.sort_values(['country', 'units'], ascending=[True, False]).groupby('country').head(5)

print("Top 5 Marcas por Unidades Vendidas:")
print(top_brands)

Top 5 Marcas por Unidades Vendidas:
      country         brand  units
2   Argentina        Chanel   1990
6   Argentina  Estée Lauder   1963
7   Argentina       L'Oréal   1958
13  Argentina          Sony   1542
14  Argentina        Xiaomi   1501
21      Chile  Estée Lauder   1317
22      Chile       L'Oréal   1270
17      Chile        Chanel   1192
29      Chile        Xiaomi   1015
23      Chile          Nike   1008
37   Colombia       L'Oréal   2571
32   Colombia        Chanel   2478
36   Colombia  Estée Lauder   2430
44   Colombia        Xiaomi   2009
43   Colombia          Sony   1950
51     España  Estée Lauder   4096
47     España        Chanel   4069
52     España       L'Oréal   4011
58     España          Sony   3379
59     España        Xiaomi   3282
67     México       L'Oréal   6688
66     México  Estée Lauder   6598
62     México        Chanel   6402
74     México        Xiaomi   5240
73     México          Sony   5046


In [29]:
# Correlación entre Rating y Unidades vendidas
popularity = df_analysis.groupby('product_name').agg({
    'units': 'sum',
    'product_rating': 'first',
    'category': 'first'
}).sort_values(by='units', ascending=False)

print("Productos más vendidos y su calificación:")
print(popularity.head(10))

Productos más vendidos y su calificación:
                              units  product_rating category
product_name                                                
Chanel Chanel No. 5            4779             4.8   Beauty
Estée Lauder Hyaluronic Acid   4194             4.2   Beauty
Chanel Hyaluronic Acid         3372             3.6   Beauty
L'Oréal Retinol Cream          3270             4.9   Beauty
Chanel Retinol Cream           3259             4.8   Beauty
L'Oréal Acqua di Gio           3215             3.5   Beauty
L'Oréal Sauvage                3153             4.3   Beauty
Estée Lauder Chanel No. 5      2899             4.8   Beauty
Estée Lauder Sauvage           2835             3.6   Beauty
L'Oréal Vitamin C              2826             3.8   Beauty


In [30]:
logistica = df_analysis.copy()
logistica['total_weight'] = logistica['units'] * logistica['weight_kg']

vendor_load = logistica.groupby('vendor')['total_weight'].sum().sort_values(ascending=False)

print("Carga Logística Total por Proveedor (kg):")
print(vendor_load)

Carga Logística Total por Proveedor (kg):
vendor
Global Home Solutions    269931.47
TechLogistics Corp       130495.71
Beauty Essentials Ltd    127901.20
Style & Wear Co.         110717.00
Name: total_weight, dtype: float64


## __Dimensión: Customers__

In [31]:

# 1. Obtener los user_id ÚNICOS de la tabla de hechos (Condición obligatoria)
user_ids = df_final['user_id'].dropna().unique()

# 2. Diccionarios de nombres y apellidos latinos
nombres_m = ['Ximena', 'Valentina', 'Mariana', 'Camila', 'Lucía', 'Fernanda', 'Gabriela', 'Sofía', 'Daniela', 'Paola']
nombres_h = ['Santiago', 'Mateo', 'Sebastián', 'Alejandro', 'Diego', 'Nicolás', 'Joaquín', 'Andrés', 'Felipe', 'Matías']
apellidos = ['González', 'Rodríguez', 'Gómez', 'Fernández', 'López', 'Díaz', 'Martínez', 'Pérez', 'García', 'Sánchez', 'Romero', 'Álvarez', 'Torres', 'Ruiz', 'Ramírez']

# 3. Función para generar fecha de registro progresiva (Marzo a Diciembre 2025)
def generate_progressive_date():
    # Pesos progresivos para los meses (Marzo=1, ..., Diciembre=10)
    months = list(range(3, 13))
    weights = [i**2 for i in range(1, 11)] # Crecimiento cuadrático hacia diciembre
    month = random.choices(months, weights=weights)[0]
    day = random.randint(1, 28)
    return datetime(2025, month, day).strftime('%Y-%m-%d')

# 4. Construcción de registros
customer_rows = []

for uid in user_ids:
    # Género: 65% Mujeres, 35% Hombres
    gender = random.choices(['Female', 'Male'], weights=[65, 35])[0]
    
    # Nombre según género
    first_name = random.choice(nombres_m) if gender == 'Female' else random.choice(nombres_h)
    last_name = random.choice(apellidos)
    
    # Email: nombre + id + dominio (60% gmail, 30% hotmail, 10% outlook)
    domain = random.choices(['@gmail.com', '@hotmail.com', '@outlook.com'], weights=[60, 30, 10])[0]
    email = f"{first_name.lower()}.{uid}{domain}"
    
    # Edad: Promedio 35 (Distribución normal)
    age = int(np.random.normal(35, 8))
    age = max(18, min(75, age)) # Límites razonables
    
    # Segmento: Mientras más VIP menos probable
    # VIP (5%), Regular (25%), New Client (70%)
    segment = random.choices(['VIP', 'Regular', 'New Client'], weights=[5, 25, 70])[0]
    
    # Acquisition Channel
    channel = random.choice(['Facebook Ads', 'Google Search', 'Organic', 'Instagram', 'TikTok Ads'])
    
    # Newsletter: 40% es 1, resto 0
    newsletter = random.choices([1, 0], weights=[40, 60])[0]
    
    customer_rows.append([
        uid, first_name, last_name, email, gender, age, 
        generate_progressive_date(), segment, channel, newsletter
    ])

# 5. Crear DataFrame
df_customers = pd.DataFrame(customer_rows, columns=[
    'user_id', 'first_name', 'last_name', 'email', 'gender', 'age', 
    'registration_date', 'customer_segment', 'acquisition_channel', 'is_newsletter_subscriber'
])

# Guardar CSV
df_customers.to_csv('dim_customers.csv', index=False)

print(f"✅ dim_customers.csv generado con {len(df_customers):,} registros únicos.")
print(f"Distribución de Género: {df_customers['gender'].value_counts(normalize=True).mul(100).round(1).to_dict()}%")
print(f"Edad Promedio: {df_customers['age'].mean():.2f} años") 

✅ dim_customers.csv generado con 89,735 registros únicos.
Distribución de Género: {'Female': 65.2, 'Male': 34.8}%
Edad Promedio: 34.60 años


In [32]:
df_customers.head(10)

,user_id,first_name,last_name,email,gender,age,registration_date,customer_segment,acquisition_channel,is_newsletter_subscriber
0,17176,Daniela,López,daniela.17176@hotmail.com,Female,37,2025-09-25,VIP,Instagram,0
1,66728,Nicolás,Torres,nicolás.66728@outlook.com,Male,41,2025-11-07,Regular,Instagram,1
2,63035,Gabriela,Martínez,gabriela.63035@gmail.com,Female,31,2025-11-02,Regular,Organic,0
3,63157,Camila,Sánchez,camila.63157@gmail.com,Female,46,2025-08-17,New Client,TikTok Ads,0
4,44764,Felipe,Rodríguez,felipe.44764@gmail.com,Male,35,2025-11-25,Regular,Facebook Ads,0
5,10543,Fernanda,García,fernanda.10543@hotmail.com,Female,42,2025-10-10,Regular,Google Search,1
6,24840,Valentina,López,valentina.24840@gmail.com,Female,37,2025-09-24,New Client,Google Search,1
7,90173,Sebastián,Martínez,sebastián.90173@hotmail.com,Male,41,2025-10-01,VIP,Google Search,0
8,62016,Sofía,Gómez,sofía.62016@gmail.com,Female,33,2025-10-26,Regular,Google Search,1
9,55188,Mariana,García,mariana.55188@hotmail.com,Female,18,2025-11-10,New Client,TikTok Ads,1


### __Validación: Dimensión customers__

In [33]:
def master_integrity_check(fact, products, customers):
    print("="*60)
    print("🛡️ AUDITORÍA DE INTEGRIDAD: STAR SCHEMA")
    print("="*60)
    
    # 1. Check de Clientes
    fact_users = set(fact['user_id'].unique())
    cust_users = set(customers['user_id'].unique())
    missing_cust = fact_users - cust_users
    
    # 2. Check de Productos
    fact_skus = set(fact['sku'].dropna().unique())
    prod_skus = set(products['sku'].unique())
    missing_prod = fact_skus - prod_skus
    
    print(f"👤 Clientes en Hechos sin perfil en Dimensión: {len(missing_cust)}")
    print(f"📦 Productos en Hechos sin descripción en Catálogo: {len(missing_prod)}")
    
    if len(missing_cust) == 0 and len(missing_prod) == 0:
        print("\n✅ INTEGRIDAD TOTAL: El esquema de estrella es consistente.")
    else:
        print("\n⚠️ ADVERTENCIA: Existen inconsistencias en las llaves foráneas.")
    print("="*60)

master_integrity_check(df_final, df_products, df_customers)

🛡️ AUDITORÍA DE INTEGRIDAD: STAR SCHEMA
👤 Clientes en Hechos sin perfil en Dimensión: 0
📦 Productos en Hechos sin descripción en Catálogo: 0

✅ INTEGRIDAD TOTAL: El esquema de estrella es consistente.


In [34]:
# Unimos las 3 tablas
full_star = df_final[df_final['event_name'] == 'purchase'].merge(df_products, on='sku').merge(df_customers, on='user_id')

analysis_1 = full_star[full_star['category'] == 'Electronics'].groupby(['country', 'gender']).agg({
    'age': 'mean',
    'units': 'sum'
}).round(1)

print("📊 Perfil de comprador de Electrónica por País:")
print(analysis_1)

📊 Perfil de comprador de Electrónica por País:
                   age  units
country   gender             
Argentina Female  34.4   3782
          Male    34.9   1938
Chile     Female  34.2   2458
          Male    34.6   1275
Colombia  Female  34.4   4618
          Male    34.5   2493
España    Female  34.5   7992
          Male    34.5   4174
México    Female  34.7  12503
          Male    34.5   6503


In [35]:
# Análisis de peso total por canal de adquisición
#UN COMENTARIO NUEVO
analysis_2 = full_star.groupby('acquisition_channel').agg({
    'weight_kg': 'sum',
    'user_id': 'nunique'
}).rename(columns={'user_id': 'total_customers'}).sort_values(by='weight_kg', ascending=False)

print("\n📈 Peso logístico movido por canal de captación:")
print(analysis_2)


📈 Peso logístico movido por canal de captación:
                     weight_kg  total_customers
acquisition_channel                            
Organic               87564.96            11852
Google Search         85121.25            11684
Facebook Ads          84977.78            11557
Instagram             84099.28            11519
TikTok Ads            83952.66            11551


In [36]:
# Registros en diciembre
reg_dic = df_customers[df_customers['registration_date'].str.contains('-12-')].groupby('customer_segment').size()

# Compras de esos mismos clientes
compras_dic = full_star[full_star['registration_date'].str.contains('-12-')].groupby('customer_segment')['units'].sum()

print("\n🎯 Desempeño de Diciembre por Segmento:")
print(pd.DataFrame({'Registros': reg_dic, 'Unidades Compradas': compras_dic}))


🎯 Desempeño de Diciembre por Segmento:
                  Registros  Unidades Compradas
customer_segment                               
New Client            16229               33405
Regular                5914               12182
VIP                    1208                2395
